# Kokoro TTS — Rig Veda Audio Generator

Generates one MP3 audio file per hymn for all **1,028 hymns** across 10 Books (Mandalas)
of the **Rig Veda** using **Kokoro PyTorch on a T4 GPU**.

Output files are named `rigveda/book-{NN}/hymn-{NNN}.mp3` and are ready for upload to
the `ankurs-books-assets` Tigris bucket via the Phase 2 upload script.

---

## Why PyTorch, not kokoro-onnx?

`kokoro-onnx` uses ONNX Runtime GPU, which pre-allocates the full model + execution
buffers into VRAM at load time — causing an **OOM crash** on Colab T4 before
generating a single word. The `kokoro` PyTorch package uses lazy CUDA allocation
and streams, which avoids OOM entirely and runs at **30–100× realtime** on T4.

## Before you start

1. **Set runtime to T4 GPU** → Runtime → Change runtime type → T4 GPU
2. **Create `data.zip` locally from the project root:**
   ```bash
   cd /path/to/ankurs-books-frontend
   zip -r /tmp/data.zip public/static/ --include '*.json'
   ```
3. Run all cells top-to-bottom. **No runtime restart needed.**
4. The notebook is **resume-safe** — if the session disconnects, re-run
   Cells 1, 3, 4 (skip Cell 2 if already uploaded). It will skip hymns
   whose MP3 already exists.

## After downloading `audio_output.zip`

The zip contains `rigveda/book-{NN}/hymn-{NNN}.mp3` at the root (no `audio/` prefix):

```bash
unzip ~/Downloads/audio_output.zip -d public/audio/
# Creates: public/audio/rigveda/book-01/hymn-001.mp3  etc.
bun run upload-audio      # Phase 2: upload MP3s to Tigris
bun run patch-json        # Phase 2: patch audio_url into JSON files
```

---

| Setting | Value |
|---|---|
| Voice | `af_heart` (American English female) |
| Speed | `1.0` |
| Sample rate | 24,000 Hz |
| Max chars/chunk | 300 |
| Output | `audio/rigveda/book-{NN}/hymn-{NNN}.mp3` per hymn |
| Total hymns | 1,028 (Books 1–10) |
| Resume | Skips hymns with existing MP3s — safe to re-run |
| Expected time | ~3–6 hours for all 1,028 hymns on T4 GPU |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1 — Install dependencies
#
# Colab T4 already has PyTorch + CUDA pre-installed.
# We only need the kokoro PyTorch TTS library and soundfile.
# NO onnxruntime-gpu, NO runtime restart required.
# ═══════════════════════════════════════════════════════════════

import subprocess, sys

print("Installing kokoro (PyTorch TTS) + soundfile...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'kokoro', 'soundfile'], check=True)
print("  Done")

print("Installing ffmpeg...")
subprocess.run(['apt-get', 'install', '-q', '-y', 'ffmpeg'],
               capture_output=True)
print("  Done")

# ── Verify ────────────────────────────────────────────────────
import torch, shutil

cuda_ok   = torch.cuda.is_available()
ffmpeg_ok = shutil.which('ffmpeg') is not None

print()
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {'✓ YES — ' + torch.cuda.get_device_name(0) if cuda_ok else '✗ NO — check runtime type'}")
print(f"ffmpeg          : {'✓' if ffmpeg_ok else '✗ not found'}")

if not cuda_ok:
    print()
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
else:
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM            : {vram:.1f} GB")
    print()
    print("✓ Ready — continue to Cell 2")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2 — Upload the Rig Veda JSON files
#
# Upload data.zip which contains the 5 JSON files from
# public/static/ (rigveda_modern-1-2.json through -9-10.json).
#
# Create it locally:
#   zip -r /tmp/data.zip public/static/ --include '*.json'
#
# Alternatively, you can upload the 5 JSON files individually.
# ═══════════════════════════════════════════════════════════════

import os, zipfile, shutil, json
from pathlib import Path
from google.colab import files

DATA_DIR   = Path('/content/data')
OUTPUT_DIR = Path('/content/audio')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = [
    'rigveda_modern-1-2.json',
    'rigveda_modern-3-4.json',
    'rigveda_modern-5-6.json',
    'rigveda_modern-7-8.json',
    'rigveda_modern-9-10.json',
]

print("Upload your data.zip (or individual JSON files).")
print("Create locally: zip -r /tmp/data.zip public/static/ --include '*.json'")
print()
uploaded = files.upload()

for filename, content in uploaded.items():
    if filename.endswith('.zip'):
        print(f"\nUnzipping {filename}...")
        tmp_extract = Path('/content/_zip_tmp')
        if tmp_extract.exists():
            shutil.rmtree(str(tmp_extract))
        tmp_extract.mkdir()

        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall(str(tmp_extract))

        # Find all JSON files matching our pattern
        found_jsons = list(tmp_extract.rglob('rigveda_modern-*.json'))
        print(f"Found {len(found_jsons)} Rig Veda JSON files in zip")
        for jf in found_jsons:
            dest = DATA_DIR / jf.name
            shutil.copy(str(jf), str(dest))
            print(f"  Copied: {jf.name}")

        shutil.rmtree(str(tmp_extract))
    elif filename.endswith('.json'):
        dest = DATA_DIR / filename
        dest.write_bytes(content)
        print(f"  Saved: {filename}")

# ── Verify all 5 files are present ────────────────────────────
print()
all_present = True
total_hymns = 0
for fname in EXPECTED_FILES:
    path = DATA_DIR / fname
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        books = data.get('books', [])
        hymn_count = sum(len(b.get('hymns', [])) for b in books)
        total_hymns += hymn_count
        book_names = ', '.join(b['title'].replace('Rig-Veda, ', '') for b in books)
        print(f"  ✓ {fname}  ({len(books)} books: {book_names}, {hymn_count} hymns)")
    else:
        print(f"  ✗ MISSING: {fname}")
        all_present = False

print()
if all_present:
    print(f"✓ All 5 JSON files loaded — {total_hymns} hymns total")
    print("Continue to Cell 3")
else:
    print("✗ Some files are missing — upload them before continuing")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3 — Define all generation functions
# ═══════════════════════════════════════════════════════════════

import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

DATA_DIR   = Path('/content/data')
OUTPUT_DIR = Path('/content/audio')

VOICE       = 'af_heart'   # American English female
SPEED       = 1.0
MAX_CHARS   = 300
SAMPLE_RATE = 24_000


# ── Text cleaning: prepare hymn content for TTS ───────────────
# Hymn content uses verse numbers like "1 First verse text\n2 Second verse..."
# We strip the bare leading numbers ("1 ", "2 ", etc.) so the TTS
# does not read them aloud, and replace line breaks between verses
# with spaces so each verse flows naturally.
def clean_hymn_text(content: str) -> str:
    """Clean raw hymn content string for TTS input."""
    lines = content.split('\n')
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Strip bare verse numbers at the start ("1 ", "12 ", "1. ", "12. ")
        line = re.sub(r'^\d{1,3}\.?\s+', '', line)
        if line:
            cleaned_lines.append(line)
    # Join lines, collapsing multiple spaces
    text = ' '.join(cleaned_lines)
    text = re.sub(r'[ \t]+', ' ', text)
    # Normalize apostrophes / fancy quotes that confuse TTS
    text = text.replace('\u2019', "'").replace('\u2018', "'") \
               .replace('\u201c', '"').replace('\u201d', '"') \
               .replace('\u2014', ' — ').replace('\u2013', ' – ')
    return text.strip()


# ── Chunking: split long text at sentence boundaries ──────────
def chunk_text(text: str, max_chars: int = MAX_CHARS) -> list:
    """Split text into chunks of at most max_chars at sentence boundaries."""
    if not text.strip():
        return []
    sentences = re.compile(r'(?<=[.!?])\s+').split(text.strip())
    chunks, current = [], ''
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        candidate = (current + ' ' + sentence).strip() if current else sentence
        if len(candidate) <= max_chars:
            current = candidate
        elif current:
            chunks.append(current)
            current = ''
            if len(sentence) <= max_chars:
                current = sentence
            else:
                # Hard-split long sentence at word boundaries
                while len(sentence) > max_chars:
                    split_at = sentence.rfind(' ', 0, max_chars)
                    if split_at == -1:
                        split_at = max_chars
                    chunks.append(sentence[:split_at])
                    sentence = sentence[split_at:].strip()
                current = sentence
        else:
            while len(sentence) > max_chars:
                split_at = sentence.rfind(' ', 0, max_chars)
                if split_at == -1:
                    split_at = max_chars
                chunks.append(sentence[:split_at])
                sentence = sentence[split_at:].strip()
            current = sentence
    if current.strip():
        chunks.append(current.strip())
    return [c for c in chunks if c.strip()]


# ── TTS: one chunk → WAV using kokoro PyTorch pipeline ────────
# _pipeline is assigned in Cell 4 after KPipeline is initialised.
_pipeline = None

def generate_chunk_wav(chunk: str, out_path: Path) -> None:
    """Run Kokoro TTS on a single text chunk and save as WAV."""
    import soundfile as sf
    import numpy as np

    segments = []
    for _, _, audio in _pipeline(chunk, voice=VOICE, speed=SPEED):
        if audio is not None and len(audio) > 0:
            segments.append(audio)

    if not segments:
        raise RuntimeError(f'No audio generated for chunk: {chunk[:60]}')

    samples = np.concatenate(segments)
    sf.write(str(out_path), samples, SAMPLE_RATE)


# ── ffmpeg: concatenate WAVs → MP3 ────────────────────────────
def concat_wavs_to_mp3(wav_paths: list, output_mp3: Path) -> None:
    """Concatenate a list of WAV files into a single MP3 via ffmpeg."""
    output_mp3.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt',
                                      delete=False, encoding='utf-8') as f:
        list_file = f.name
        for p in wav_paths:
            f.write(f"file '{str(p)}'\n")
    try:
        result = subprocess.run(
            ['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
             '-i', list_file, '-c:a', 'libmp3lame', '-q:a', '4',
             str(output_mp3)],
            capture_output=True, text=True, timeout=600,
        )
        if result.returncode != 0:
            raise RuntimeError(f'ffmpeg error: {result.stderr[-400:]}')
    finally:
        os.unlink(list_file)


# ── Process one hymn: text → chunks → WAVs → MP3 ─────────────
def process_hymn(hymn: dict, output_mp3: Path,
                  force: bool = False,
                  hymn_num: int = 0, total: int = 0) -> bool:
    """
    Generate audio for a single hymn.
    Returns True if MP3 was generated, False if skipped.
    """
    label = output_mp3.relative_to(OUTPUT_DIR)
    pfx   = f'[{hymn_num}/{total}] ' if total else ''

    # Resume-safe: skip if MP3 already exists
    if output_mp3.exists() and not force:
        print(f'  SKIP  {label}')
        return False

    raw_content = hymn.get('content', '').strip()
    if not raw_content:
        print(f'  SKIP  {label}  (empty content)')
        return False

    # Prepend the hymn title so the audio is self-contained
    hymn_title = hymn.get('title', '').strip()
    # Clean title: e.g. "HYMN I. Agni." → spoken naturally
    # Convert Roman numeral hymn numbers in the title for TTS
    title_for_tts = re.sub(r'HYMN\s+([IVXLCDM]+)\.?', '', hymn_title).strip()
    title_for_tts = re.sub(r'^[.\s]+', '', title_for_tts).strip()

    text = clean_hymn_text(raw_content)
    if title_for_tts:
        text = f"{title_for_tts}. {text}"

    chunks = chunk_text(text)
    if not chunks:
        print(f'  SKIP  {label}  (no chunks after cleaning)')
        return False

    print(f'\n{pfx}{label}  "{hymn_title}"  ({len(text):,} chars → {len(chunks)} chunks)')
    hymn_start = time.perf_counter()

    with tempfile.TemporaryDirectory(prefix='kokoro-') as tmp:
        tmp_dir   = Path(tmp)
        wav_files = []

        for i, chunk in enumerate(chunks, 1):
            chunk_wav = tmp_dir / f'chunk_{i:04d}.wav'
            t0 = time.perf_counter()
            try:
                generate_chunk_wav(chunk, chunk_wav)
            except Exception as e:
                print(f'  ✗ Chunk {i}/{len(chunks)}: {e}')
                return False

            elapsed = time.perf_counter() - t0
            preview = chunk[:60].replace('\n', ' ') + ('...' if len(chunk) > 60 else '')
            print(f'  [{i:02d}/{len(chunks)}] ({elapsed:.2f}s) "{preview}"')
            wav_files.append(chunk_wav)

        print('  Concatenating → MP3...')
        try:
            concat_wavs_to_mp3(wav_files, output_mp3)
        except Exception as e:
            print(f'  ✗ ffmpeg: {e}')
            return False

    elapsed_total = time.perf_counter() - hymn_start
    size_kb = output_mp3.stat().st_size // 1024
    print(f'  ✓ {size_kb} KB  {elapsed_total:.1f}s')
    return True


# ── Build flat list of all hymns from the 5 JSON files ────────
def discover_hymns() -> list:
    """
    Returns a flat list of dicts:
      { book_num, hymn_num, hymn_index, hymn, output_mp3 }
    sorted by book then hymn order.

    Naming convention (rigveda namespace for multi-book library):
      audio/rigveda/book-{NN}/hymn-{NNN}.mp3
      e.g. audio/rigveda/book-01/hymn-001.mp3
    """
    SOURCE_FILES = [
        'rigveda_modern-1-2.json',
        'rigveda_modern-3-4.json',
        'rigveda_modern-5-6.json',
        'rigveda_modern-7-8.json',
        'rigveda_modern-9-10.json',
    ]

    items = []
    for fname in SOURCE_FILES:
        path = DATA_DIR / fname
        if not path.exists():
            print(f'  ⚠ Missing: {fname} — run Cell 2 first')
            continue
        data = json.loads(path.read_text(encoding='utf-8'))
        for book in data.get('books', []):
            # Extract book number from title e.g. "Rig-Veda, Book 1"
            m = re.search(r'Book\s+(\d+)', book.get('title', ''))
            book_num = int(m.group(1)) if m else 0
            for hymn_idx, hymn in enumerate(book.get('hymns', []), 1):
                # rigveda/ namespace keeps Rig Veda separate from future books
                output_mp3 = OUTPUT_DIR / 'rigveda' / f'book-{book_num:02d}' / f'hymn-{hymn_idx:03d}.mp3'
                items.append({
                    'book_num':   book_num,
                    'hymn_idx':   hymn_idx,
                    'book_title': book.get('title', ''),
                    'hymn':       hymn,
                    'output_mp3': output_mp3,
                })

    items.sort(key=lambda x: (x['book_num'], x['hymn_idx']))
    return items


# Quick sanity check
all_hymns = discover_hymns()
already_done = sum(1 for h in all_hymns if h['output_mp3'].exists())

print(f'✓ Functions defined')
print(f'  Total hymns   : {len(all_hymns)}')
print(f'  Already done  : {already_done} MP3s exist')
print(f'  To generate   : {len(all_hymns) - already_done}')
print()
# Show breakdown by book
from collections import Counter
book_counts = Counter(h['book_num'] for h in all_hymns)
done_counts = Counter(h['book_num'] for h in all_hymns if h['output_mp3'].exists())
print('Book breakdown:')
for bk in sorted(book_counts):
    total_bk = book_counts[bk]
    done_bk  = done_counts.get(bk, 0)
    bar = '█' * done_bk + '░' * (total_bk - done_bk)
    print(f'  Book {bk:>2}: {done_bk:>3}/{total_bk:<3}  {bar[:40]}')
print()
print('Continue to Cell 4')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4 — Initialise Kokoro PyTorch pipeline and generate audio
#
# KPipeline downloads ~500MB of model weights from HuggingFace
# on the first run (cached after that). PyTorch uses lazy CUDA
# allocation so there is NO VRAM OOM on Colab T4.
#
# Resume-safe: hymns with existing MP3s are skipped automatically.
# Estimated time: ~3–6 hours for all 1,028 hymns on T4 GPU.
#
# CONFIGURATION — adjust if needed:
BOOK_FILTER = None  # None = all books; or e.g. [1, 2] to only do Books 1 & 2
FORCE_REGEN = False # True = regenerate even if MP3 already exists
# ═══════════════════════════════════════════════════════════════

import time, torch
from kokoro import KPipeline

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU check ─────────────────────────────────────────────────
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  No GPU — running on CPU (very slow, expect 10–20× longer)')
    print('   Go to Runtime → Change runtime type → T4 GPU')

# ── Load Kokoro pipeline ──────────────────────────────────────
print('\nLoading Kokoro pipeline...')
print('(First run downloads ~500MB model from HuggingFace — takes 2-3 min)')
t0 = time.time()
_pipeline = KPipeline(lang_code='a')  # 'a' = American English (for af_heart)
print(f'✓ Pipeline ready in {time.time()-t0:.1f}s')

# ── Speed benchmark ───────────────────────────────────────────
print('\nRunning speed benchmark...')
test_text = 'Agni, the chosen Priest, God, minister of sacrifice, the richest in wealth.'
t0 = time.time()
for _, _, audio in _pipeline(test_text, voice=VOICE, speed=SPEED):
    pass
bench = time.time() - t0

device = 'GPU' if torch.cuda.is_available() else 'CPU'
# Rough estimate: avg hymn ≈ 500 chars ≈ 2 chunks ≈ bench×2 seconds
est_hrs = (len(all_hymns) * 2 * bench) / 3600
print(f'✓ Benchmark: {bench:.3f}s for test chunk ({device})')
print(f'  Estimated total: ~{est_hrs:.1f} hours for {len(all_hymns)} hymns')
if bench > 2.0 and device == 'GPU':
    print('  Note: first chunk is slower (model warming up) — subsequent chunks will be faster')

# ── Filter hymns if BOOK_FILTER is set ───────────────────────
hymns_to_run = all_hymns
if BOOK_FILTER:
    hymns_to_run = [h for h in all_hymns if h['book_num'] in BOOK_FILTER]
    print(f'\n⚠ BOOK_FILTER active: only generating Books {BOOK_FILTER}')

already_done = sum(1 for h in hymns_to_run if h['output_mp3'].exists())
to_generate  = len(hymns_to_run) - already_done

print(f'\nHymns to process : {len(hymns_to_run)}')
print(f'Already done     : {already_done} (skipping)')
print(f'To generate      : {to_generate}')
print('─' * 50)

if to_generate == 0:
    print('\n✓ All hymns already generated! Run Cell 5 to zip and download.')
else:
    # ── Generate ─────────────────────────────────────────────
    processed = skipped = failed = 0
    run_start = time.time()

    for i, item in enumerate(hymns_to_run, 1):
        try:
            did = process_hymn(
                hymn=item['hymn'],
                output_mp3=item['output_mp3'],
                force=FORCE_REGEN,
                hymn_num=i,
                total=len(hymns_to_run),
            )
            if did: processed += 1
            else:   skipped   += 1
        except KeyboardInterrupt:
            print('\n⚠ Interrupted — partial results saved in /content/audio/')
            print('Re-run Cell 4 to resume from where you left off.')
            break
        except Exception as e:
            print(f'  ✗ Book {item["book_num"]} Hymn {item["hymn_idx"]}: {e}')
            failed += 1

    # ── Summary ──────────────────────────────────────────────
    wall  = time.time() - run_start
    mp3s  = len(list(OUTPUT_DIR.rglob('*.mp3')))
    print('\n' + '═' * 50)
    print('Generation complete')
    print(f'  Processed  : {processed}')
    print(f'  Skipped    : {skipped} (already existed)')
    print(f'  Failed     : {failed}')
    print(f'  Wall time  : {wall:.0f}s ({wall/60:.1f} min)')
    print(f'  MP3 files  : {mp3s} total in /content/audio/')
    print('═' * 50)
    print('\nNext: run Cell 5 to zip, then Cell 6 to download')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5 — Verify output structure
#
# Optional but recommended — run this to confirm all expected
# MP3s are present and check for any missing hymns before zipping.
# ═══════════════════════════════════════════════════════════════

from pathlib import Path
from collections import defaultdict

OUTPUT_DIR  = Path('/content/audio')
RIGVEDA_DIR = OUTPUT_DIR / 'rigveda'  # namespace subfolder

mp3s = sorted(OUTPUT_DIR.rglob('*.mp3'))
print(f'Total MP3 files: {len(mp3s)}')
print()

# Group by book — mp3.parent is rigveda/book-{NN}, so use parent.name
by_book = defaultdict(list)
for mp3 in mp3s:
    book_dir = mp3.parent.name  # e.g. "book-01"
    by_book[book_dir].append(mp3.name)

# Expected counts per book (from JSON structure)
EXPECTED_COUNTS = {1: 191, 2: 43, 3: 62, 4: 58, 5: 87, 6: 75, 7: 104, 8: 103, 9: 114, 10: 191}

all_ok = True
print('Book-level verification (looking in rigveda/ subfolder):')
for bk_num, expected in sorted(EXPECTED_COUNTS.items()):
    bk_key = f'book-{bk_num:02d}'
    bk_dir = RIGVEDA_DIR / bk_key
    actual = len(list(bk_dir.glob('*.mp3'))) if bk_dir.exists() else 0
    status = '✓' if actual == expected else f'✗ missing {expected - actual}'
    if actual != expected:
        all_ok = False
    print(f'  Book {bk_num:>2} (rigveda/{bk_key}): {actual:>3}/{expected:<3}  {status}')

print()
total_expected = sum(EXPECTED_COUNTS.values())
print(f'Total: {len(mp3s)}/{total_expected} hymns')

if all_ok:
    print()
    print('✓ All hymns generated! Continue to Cell 6 to zip.')
else:
    print()
    print('⚠ Some hymns are missing. You can still zip and upload what you have,')
    print('  then re-run Cell 4 to fill in the gaps later.')

# Show any missing hymns
if not all_ok and len(all_hymns) > 0:
    missing = [h for h in all_hymns if not h['output_mp3'].exists()]
    print(f'\nFirst 10 missing hymns:')
    for h in missing[:10]:
        print(f"  Book {h['book_num']} Hymn {h['hymn_idx']}: {h['hymn'].get('title', '')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6 — Zip the audio/ output folder
# ═══════════════════════════════════════════════════════════════

import shutil, zipfile
from pathlib import Path

OUTPUT_DIR = Path('/content/audio')
ZIP_PATH   = Path('/content/audio_output.zip')

mp3s = list(OUTPUT_DIR.rglob('*.mp3'))

if not mp3s:
    print('✗ No MP3 files in /content/audio/ — run Cell 4 first')
else:
    print(f'Zipping {len(mp3s)} MP3 files...')
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    # KEY: root_dir=OUTPUT_DIR so the zip contains
    # rigveda/book-{NN}/hymn-{NNN}.mp3 at the ROOT (no 'audio/' prefix).
    # Locally: unzip audio_output.zip -d public/audio/
    # gives:  public/audio/rigveda/book-01/hymn-001.mp3  ✓
    shutil.make_archive(
        base_name=str(ZIP_PATH.with_suffix('')),
        format='zip',
        root_dir=str(OUTPUT_DIR),
        base_dir='.',
    )
    size_mb = ZIP_PATH.stat().st_size / 1024 / 1024
    print(f'✓ audio_output.zip  ({size_mb:.1f} MB)')
    print(f'  MP3 files: {len(mp3s)}')

    # Verify zip structure
    with zipfile.ZipFile(str(ZIP_PATH)) as zf:
        first_mp3s = [n for n in zf.namelist() if n.endswith('.mp3')][:5]
    print(f'\nZip structure (should be rigveda/book-{{NN}}/hymn-{{NNN}}.mp3):')
    for e in first_mp3s:
        print(f'  {e}')

    # Summary by book
    rigveda_dir = OUTPUT_DIR / 'rigveda'
    books_present = sorted(set(mp3.parent.name for mp3 in mp3s if mp3.parent.parent.name == 'rigveda'))
    print(f'\nBooks ({len(books_present)}):')
    for bk in books_present:
        n = len(list((rigveda_dir / bk).glob('*.mp3')))
        print(f'  rigveda/{bk}/  ({n} hymns)')

    print('\nRun Cell 7 to download')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7 — Download audio_output.zip
#
# The zip contains rigveda/book-{NN}/hymn-{NNN}.mp3 at the root.
# After downloading, run locally:
#
#   unzip ~/Downloads/audio_output.zip -d public/audio/
#   # Creates: public/audio/rigveda/book-01/hymn-001.mp3  etc.
#
#   # Upload to Tigris and patch JSON files
#   bun run upload-audio
#   bun run patch-json
# ═══════════════════════════════════════════════════════════════

from google.colab import files
from pathlib import Path

ZIP_PATH = Path('/content/audio_output.zip')

if not ZIP_PATH.exists():
    print('✗ audio_output.zip not found — run Cell 6 first')
else:
    size_mb = ZIP_PATH.stat().st_size / 1024 / 1024
    print(f'Downloading audio_output.zip ({size_mb:.1f} MB)...')
    print('(Large files may take a few minutes to appear in your Downloads folder)')
    files.download(str(ZIP_PATH))
    print()
    print('✓ Download started')
    print()
    print('Next steps locally (from the ankurs-books-frontend project root):')
    print()
    print('  # 1. Extract the MP3s (creates public/audio/rigveda/book-01/hymn-001.mp3 etc.)')
    print('  unzip ~/Downloads/audio_output.zip -d public/audio/')
    print()
    print('  # 2. Upload to Tigris under audio/rigveda/ prefix')
    print('  bun run upload-audio')
    print()
    print('  # 3. Re-patch audio_url in CDN JSON files with new paths')
    print('  bun run patch-json')